# Project 05: PySpark Fundamentals & Core DataFrames
## Data Engineering Diaries by Sathiya Murthy

This notebook demonstrates production-grade PySpark 3.5+ concepts including:
1. **SparkSession Lifecycle & Configuration**
2. **DataFrame Ingestion with Explicit StructType Schemas**
3. **Core Transformations (select, filter, withColumn, unionByName)**
4. **Multi-Dimensional Aggregations & Pivot Tables**
5. **Window Functions (Ranking, Lead/Lag, Running Totals)**
6. **Join Strategies (Broadcast Hash Join, Semi/Anti Joins)**
7. **Data Quality Checks & Parquet Export**

In [ ]:
import os
import sys

# Add project root directory to path
sys.path.insert(0, os.path.abspath(".."))

from src.spark_session import SparkSessionFactory

# Initialize local SparkSession
spark = SparkSessionFactory.create_local(
    app_name="NotebookPySparkFundamentals",
    extra_config={
        "spark.sql.shuffle.partitions": "4",
        "spark.sql.adaptive.enabled": "true",
    },
    log_level="WARN",
)

print(f"Spark Version: {spark.version}")
print(f"Master: {spark.conf.get('spark.master')}")
print(f"Shuffle Partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

## 1. Data Ingestion with Explicit StructType Schemas
Reading heterogeneous data formats (CSV, JSON) using pre-defined StructType schemas to avoid inference latency.

In [ ]:
from src.dataframe_basics import SALES_SCHEMA, CUSTOMER_EVENT_SCHEMA, PRODUCT_SCHEMA, read_csv, read_json

sales_df = read_csv(spark, "../sample-data/sales_data.csv", schema=SALES_SCHEMA)
events_df = read_json(spark, "../sample-data/customer_events.json", schema=CUSTOMER_EVENT_SCHEMA)
products_df = read_csv(spark, "../sample-data/product_catalog.csv", schema=PRODUCT_SCHEMA)

print("Sales Schema:")
sales_df.printSchema()

print("Sample Sales Records:")
sales_df.show(5, truncate=False)

## 2. Filtering & Derived Monetary Metrics
Filter completed transactions and compute gross amount, discount amount, calculated total, and high-value flags.

In [ ]:
from src.transformations import filter_sales, add_derived_metrics

filtered_sales = filter_sales(sales_df, min_amount=50.0, statuses=["Completed", "Shipped"])
metrics_df = add_derived_metrics(filtered_sales)

metrics_df.select(
    "sale_id", "category", "quantity", "unit_price", "discount_pct",
    "gross_amount", "discount_amount", "calculated_total", "high_value_flag"
).show(5)

## 3. Aggregations & Pivot Tables
Demonstrates multi-dimensional grouping and pivoting sales revenue by Category across Regions.

In [ ]:
from src.aggregations import group_by_multi_dim, pivot_sales_by_region, hierarchical_rollup

print("--- Multi-Dimension GroupBy ---")
group_by_multi_dim(metrics_df, ["category", "region"]).show(5)

print("--- Pivoted Sales (Category vs Region) ---")
pivot_sales_by_region(metrics_df, pivot_col="region", pivot_values=["North", "South", "East", "West"]).show()

print("--- Hierarchical Rollup Sub-Totals ---")
hierarchical_rollup(metrics_df, ["category", "region"]).show(10)

## 4. Window Functions
Calculate regional sales rankings, lead/lag period-over-period differences, and cumulative running totals.

In [ ]:
from src.window_functions import calculate_sales_ranks, calculate_running_totals_and_moving_avg

print("--- Regional Sales Ranks ---")
calculate_sales_ranks(metrics_df).select("sale_id", "region", "total_amount", "row_num", "rank", "dense_rank").show(5)

print("--- Cumulative Revenue & Moving Average ---")
calculate_running_totals_and_moving_avg(metrics_df, window_size=3).select("sale_id", "region", "order_date", "total_amount", "cumulative_revenue", "moving_avg_revenue").show(5)

## 5. Joins & Broadcast Optimization
Enrich sales DataFrames using Broadcast Hash Join (BHJ) with product metadata, avoiding network shuffle overhead.

In [ ]:
from src.joins import execute_broadcast_join, filter_active_customers_semi_join

enriched_sales = execute_broadcast_join(metrics_df, products_df, join_key="product_id", join_type="left")

print("Enriched Sales (Broadcast Joined):")
enriched_sales.select("sale_id", "product_id", "product_name", "category", "brand", "total_amount").show(5)

# Stop Spark session
spark.stop()